# # Lab 2 — Cleaning & Normalization Pipeline
# Цей зошит демонструє роботу детермінованого пайплайну для очистки ІТ-вакансій.

In [1]:
import sys
import os
import json
from pathlib import Path
import pandas as pd

sys.path.append(os.path.abspath('..'))

from src.preprocess import preprocess, PreprocessPolicy

In [2]:
sample_path = Path('../data/sample/sample_raw.csv')

if not sample_path.exists():
    sample_path = Path('data/sample/sample_raw.csv')

df = pd.read_csv(sample_path)
print(f"Завантаженорядків: {len(df)}")
df.head(2)

Завантаженорядків: 15


,link,title,source_category,description_cleaned
0,https://jobs.dou.ua/companies/simplexoft/vacan...,Job Position: Senior Frontend / Full Stack Eng...,Python,Agentpunkt Всі вакансії компанії Agentpunkt is...
1,https://jobs.dou.ua/companies/jobtestprep/vaca...,Senior Python Software Engineer,Python,JobTestPrep Всі вакансії компанії JobTestPrep ...


In [3]:
policy = PreprocessPolicy(
    normalize_unicode=True,
    normalize_whitespace=True,
    normalize_quotes=True,
    normalize_dashes=True,
    normalize_apostrophes=True,
    normalize_homoglyphs=False, 
    mask_urls=True,
    mask_emails=True,
    mask_phones=True,
    remove_dou_boilerplate=True,
    sentence_split=True
)

text_column = 'description_cleaned' if 'description_cleaned' in df.columns else df.columns[0]

processed_results = df[text_column].astype(str).apply(lambda x: preprocess(x, policy))

df['clean_text'] = processed_results.apply(lambda d: d['clean_text'])
df['sentences'] = processed_results.apply(lambda d: d.get('sentences', []))

print("Пайплайн успішно відпрацював!")
df[[text_column, 'clean_text']].head(3)

Пайплайн успішно відпрацював!


,description_cleaned,clean_text
0,Agentpunkt Всі вакансії компанії Agentpunkt is...,Agentpunkt Всі вакансії компанії Agentpunkt is...
1,JobTestPrep Всі вакансії компанії JobTestPrep ...,JobTestPrep Всі вакансії компанії JobTestPrep ...
2,N-iX Всі вакансії компанії N-iX is a Ukrainian...,N-iX Всі вакансії компанії N-iX is a Ukrainian...


In [4]:
edge_path = Path('../tests/edge_cases.jsonl')
if not edge_path.exists():
    edge_path = Path('tests/edge_cases.jsonl')

edge_rows = [json.loads(line) for line in edge_path.read_text(encoding='utf-8').splitlines() if line.strip()]

results = []
for r in edge_rows:
    raw = r['raw_text']
    out_1 = preprocess(raw, policy)
    clean_1 = out_1['clean_text']
    
    # 1. Idempotence test: проганяємо вдруге
    out_2 = preprocess(clean_1, policy)
    clean_2 = out_2['clean_text']
    is_idempotent = (clean_1 == clean_2)
    
    # 2. No empty explosions: якщо вхід не порожній, вихід теж не має бути порожнім
    no_explosion = not (len(raw.strip()) > 0 and len(clean_1.strip()) == 0)
    
    results.append({
        'id': r['id'],
        'raw': raw,
        'clean': clean_1,
        'idempotent': is_idempotent,
        'no_explosion': no_explosion,
        'sentences': out_1.get('sentences', [])
    })

tests_df = pd.DataFrame(results)

failed_tests = tests_df[~tests_df['idempotent'] | ~tests_df['no_explosion']]
if not failed_tests.empty:
    print("УВАГА! Є провалені тести регресії:")
    display(failed_tests)
else:
    print("Усі 20 Edge Cases пройшли перевірку на Ідемпотентність та No Explosions!")

tests_df[['id', 'raw', 'clean', 'sentences']].head(5)

Усі 20 Edge Cases пройшли перевірку на Ідемпотентність та No Explosions!


,id,raw,clean,sentences
0,ec_01,Шукаємо експерта з Node.js для бекенду.,Шукаємо експерта з Node.js для бекенду.,[Шукаємо експерта з Node.js для бекенду.]
1,ec_02,Досвід з Vue.js та React.js обов'язковий.,Досвід з Vue.js та React.js обов'язковий.,[Досвід з Vue.js та React.js обов'язковий.]
2,ec_03,Розробка під .NET платформу.,Розробка під .NET платформу.,[Розробка під .NET платформу.]
3,ec_04,Пишемо на C++. Долучайся.,Пишемо на C++. Долучайся.,"[Пишемо на C++., Долучайся.]"
4,ec_05,Знання C# буде величезним плюсом.,Знання C# буде величезним плюсом.,[Знання C# буде величезним плюсом.]


In [5]:
mask_counts = {
    '<URL>': int(df['clean_text'].str.contains('<URL>', regex=False).sum()),
    '<EMAIL>': int(df['clean_text'].str.contains('<EMAIL>', regex=False).sum()),
    '<PHONE>': int(df['clean_text'].str.contains('<PHONE>', regex=False).sum()),
}

dou_removed = int(df[text_column].str.contains('Facebook Twitter LinkedIn', case=False, na=False).sum())

summary = [
    '# Lab 2 Audit Summary\n',
    '## Статистика маскування та очистки:',
    f'- Замасковано URL: **{mask_counts["<URL>"]}**',
    f'- Замасковано Email: **{mask_counts["<EMAIL>"]}**',
    f'- Замасковано Телефонів: **{mask_counts["<PHONE>"]}**',
    f'- Видалено DOU boilerplate (кнопки соцмереж): **{dou_removed}** входжень\n',
    '## Результати регресійного тестування:',
    f'- Edge cases протестовано: **{len(tests_df)}**',
    '- Idempotence (f(f(x)) == f(x)): **Успішно** (100%)',
    '- No empty explosions: **Успішно** (100%)\n',
    '## Висновки:',
    '- Уніфіковано апострофи (важливо для українського IT-суржику: Python\'ом).',
    '- Розроблено кастомний спліттер, який захищає слова з крапками (Node.js, C++).',
    '- Англійські літери не замінюються на кирилицю (Homoglyphs normalization=False), щоб не втратити назви технологій.'
]

out_path = Path('../docs/audit_summary_lab2.md')
if not out_path.exists() and Path('docs').exists():
    out_path = Path('docs/audit_summary_lab2.md')

out_path.write_text('\n'.join(summary), encoding='utf-8')
print(f"Звіт успішно згенеровано та збережено у {out_path}")

Звіт успішно згенеровано та збережено у ..\docs\audit_summary_lab2.md
